In [21]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")




Using device: cpu


In [22]:
# Load dataset
df = pd.read_csv("stroke_dataset.csv")

# Drop head coordinates (not used)
df = df.drop(columns=["head_x", "head_y"], errors="ignore")

# Quick check
df.head()


,imu1_acc_x,imu1_acc_y,imu1_acc_z,imu1_gyro_x,imu1_gyro_y,imu1_gyro_z,imu2_acc_x,imu2_acc_y,imu2_acc_z,imu2_gyro_x,...,imu9_gyro_y,imu9_gyro_z,imu10_acc_x,imu10_acc_y,imu10_acc_z,imu10_gyro_x,imu10_gyro_y,imu10_gyro_z,stroke_label,stroke_prob
0,-0.907,0.113,-1.699,0.240,0.508,-0.170,1.410,0.547,2.006,-0.184,...,-0.412,-0.224,-0.082,1.555,0.097,-0.829,0.475,-1.187,Freestyle,0.890
1,2.188,-1.561,-0.167,2.827,0.501,0.141,-0.064,0.721,-1.297,-1.100,...,0.917,0.662,2.094,0.433,-1.086,-0.966,-0.565,0.927,Front Crawl,0.731
2,0.319,0.774,-0.769,0.573,0.511,-0.729,-0.878,1.242,1.006,-1.718,...,0.503,-0.798,-0.080,0.198,-0.399,0.451,-1.144,-1.222,Front Crawl,0.320
3,-0.138,-1.061,-0.222,-0.675,-0.527,-1.631,-0.625,0.762,-1.457,0.412,...,0.752,-1.567,0.010,0.418,3.012,-0.170,-1.378,-0.992,Front Crawl,0.678
4,-0.106,-0.076,0.783,-1.040,0.457,-0.708,-0.850,0.249,-1.024,0.718,...,-0.870,-0.785,-0.044,-1.116,0.285,-2.265,-0.585,-1.062,Breaststroke,0.521


In [24]:
# Separate features and labels
X = df.drop(columns=["stroke_label", "stroke_prob"]).values
y = df["stroke_label"].values
stroke_prob = df["stroke_prob"].values  # confidence score (optional)

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)
w_tensor = torch.tensor(stroke_prob, dtype=torch.float32)

# Dataset split
dataset = TensorDataset(X_tensor, y_tensor, w_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

print(f"✅ Train samples: {train_size}, Validation samples: {val_size}")


✅ Train samples: 1608, Validation samples: 402


In [25]:
class StrokeClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

input_dim = X_tensor.shape[1]
num_classes = len(np.unique(y_encoded))
model = StrokeClassifier(input_dim, num_classes)
print(model)


StrokeClassifier(
  (layers): Sequential(
    (0): Linear(in_features=60, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=5, bias=True)
  )
)


In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("🔧 Using device:", device)

model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


🔧 Using device: cpu


In [31]:
best_val_loss = np.inf
patience = 100  # Increased patience
wait = 0
EPOCHS = 50

train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch, w_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_val, y_val, _ in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            outputs = model(X_val)
            loss = criterion(outputs, y_val)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # Early Stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        wait = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        wait += 1
        if wait >= patience:
            print("⏹️ Early stopping triggered!")
            break

Epoch 1/50 | Train Loss: 0.4009 | Val Loss: 3.4049
Epoch 2/50 | Train Loss: 0.3835 | Val Loss: 3.4241
Epoch 3/50 | Train Loss: 0.3906 | Val Loss: 3.4504
Epoch 4/50 | Train Loss: 0.4035 | Val Loss: 3.4276
Epoch 5/50 | Train Loss: 0.3699 | Val Loss: 3.5290
Epoch 6/50 | Train Loss: 0.3777 | Val Loss: 3.5414
Epoch 7/50 | Train Loss: 0.3764 | Val Loss: 3.5994
Epoch 8/50 | Train Loss: 0.3653 | Val Loss: 3.6183
Epoch 9/50 | Train Loss: 0.3808 | Val Loss: 3.6060
Epoch 10/50 | Train Loss: 0.3447 | Val Loss: 3.6533
Epoch 11/50 | Train Loss: 0.3594 | Val Loss: 3.6328
Epoch 12/50 | Train Loss: 0.3454 | Val Loss: 3.7146
Epoch 13/50 | Train Loss: 0.3513 | Val Loss: 3.6774
Epoch 14/50 | Train Loss: 0.3388 | Val Loss: 3.6847
Epoch 15/50 | Train Loss: 0.3641 | Val Loss: 3.6959
Epoch 16/50 | Train Loss: 0.3309 | Val Loss: 3.7232
Epoch 17/50 | Train Loss: 0.3351 | Val Loss: 3.7190
Epoch 18/50 | Train Loss: 0.3335 | Val Loss: 3.8093
Epoch 19/50 | Train Loss: 0.3653 | Val Loss: 3.7403
Epoch 20/50 | Train L

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.legend()
plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.grid(True)
plt.show()



In [ ]:
# Load best model
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch, _ in val_loader:
        X_batch = X_batch.to(device)
        preds = torch.argmax(model(X_batch), dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

# Classification report
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()
